In [1]:
!pip install albumentations
!pip install opencv-python

# 0. Library Importation

In [ ]:
import os
import random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split

from utils.helper import normalize_mask, export_dataset_to_disk

In [3]:
print("All libraries imported successfully")
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("OpenCV version:", cv2.__version__)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Please enable GPU in Kaggle settings.")

!nvidia-smi

All libraries imported successfully
PyTorch version: 2.10.0+cpu
Torchvision version: 0.25.0+cpu
OpenCV version: 4.13.0
CUDA available: False
Device: cpu
GPU is not available. Please enable GPU in Kaggle settings.
/bin/bash: line 1: nvidia-smi: command not found


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 1. Dataset Load

In [ ]:
# ===== PATH =====
BASE_PATH = "/kaggle/input/datasets/zionfuo/drive2004/DRIVE/"

train_img_dir = os.path.join(BASE_PATH, "training/images")
train_mask_dir = os.path.join(BASE_PATH, "training/mask")
train_manual_dir = os.path.join(BASE_PATH, "training/1st_manual")

test_img_dir = os.path.join(BASE_PATH, "test/images")
test_mask_dir = os.path.join(BASE_PATH, "test/mask")
test_manual_1_dir = os.path.join(BASE_PATH, "test/1st_manual")
test_manual_2_dir = os.path.join(BASE_PATH, "test/2nd_manual")

In [ ]:
# ===== LOAD TRAIN =====
train_images = sorted(os.listdir(train_img_dir))

train_image_paths = []
train_mask_paths = []
train_manual_paths = []

for img_name in train_images:
    img_id = img_name.split('_')[0]
    manual_name = f"{img_id}_manual1.gif"
    mask_name = f"{img_id}_training_mask.gif"
    
    train_image_paths.append(os.path.join(train_img_dir, img_name))
    train_manual_paths.append(os.path.join(train_manual_dir, manual_name))
    train_mask_paths.append(os.path.join(train_mask_dir, mask_name))

In [ ]:
# ===== SPLIT TRAIN - VAL =====
train_image_paths, val_image_paths, train_mask_paths, val_mask_paths, train_manual_paths, val_manual_paths = train_test_split(
    train_image_paths, train_mask_paths, train_manual_paths,
    test_size=4, 
    random_state=42
)

In [ ]:
# ===== LOAD TEST =====
test_images = sorted(os.listdir(test_img_dir))

test_image_paths = []
test_mask_paths = []
test_manual_1_paths = []
test_manual_2_paths = []

for img_name in test_images:
    img_id = img_name.split('_')[0]
    mask_name = f"{img_id}_test_mask.gif"
    manual_1_name = f"{img_id}_manual1.gif"
    manual_2_name = f"{img_id}_manual2.gif"
    
    test_image_paths.append(os.path.join(test_img_dir, img_name))
    test_mask_paths.append(os.path.join(test_mask_dir, mask_name))
    test_manual_1_paths.append(os.path.join(test_manual_1_dir, manual_1_name))
    test_manual_2_paths.append(os.path.join(test_manual_2_dir, manual_2_name))

# 2. Data Augment + Loader

In [ ]:
class DRIVEDataset(Dataset):
    def __init__(self, image_paths, mask_paths, manual_paths, manual_paths_2=None, img_size=(256, 256), transform=None, multiplier=1):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.manual_paths = manual_paths
        self.manual_paths_2 = manual_paths_2
        
        self.img_size = img_size
        self.transform = transform
        self.multiplier = multiplier

    def __len__(self):
        return len(self.image_paths) * self.multiplier

    def __getitem__(self, idx):
        real_idx = idx % len(self.image_paths)

        # ===== 1. LOAD DATA =====
        img_path = self.image_paths[real_idx]
        mask_path = self.mask_paths[real_idx]
        manual_path = self.manual_paths[real_idx]

        # Mở ảnh gốc (RGB) và các masks (Grayscale)
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        manual = Image.open(manual_path).convert("L")

        # Load 2nd_manual nếu đường dẫn được cung cấp
        if self.manual_paths_2 is not None:
            manual_path_2 = self.manual_paths_2[real_idx]
            manual_2 = Image.open(manual_path_2).convert("L")

        # ===== 2. RESIZE =====
        img = img.resize(self.img_size)
        mask = mask.resize(self.img_size, Image.NEAREST)
        manual = manual.resize(self.img_size, Image.NEAREST)
        
        if self.manual_paths_2 is not None:
            manual_2 = manual_2.resize(self.img_size, Image.NEAREST)

        # Ép kiểu Numpy Array
        img = np.array(img, dtype=np.float32)
        mask = np.array(mask, dtype=np.float32)
        manual = np.array(manual, dtype=np.float32)
        
        if self.manual_paths_2 is not None:
            manual_2 = np.array(manual_2, dtype=np.float32)

        # ===== 3. APPLY AUGMENTATION =====
        if self.transform is not None:
            # Gom tất cả các mask vào 1 list để albumentations biến đổi đồng bộ
            masks_list = [mask, manual]
            if self.manual_paths_2 is not None:
                masks_list.append(manual_2)
                
            # Sử dụng key 'masks' thay vì 'mask'
            augmented = self.transform(image=img, masks=masks_list)
            
            img = augmented['image']
            mask = augmented['masks'][0]
            manual = augmented['masks'][1]
            
            if self.manual_paths_2 is not None:
                manual_2 = augmented['masks'][2]

        # ===== 4. NORMALIZE & TENSOR CONVERSION =====
        img = img / 255.0 
        mask = mask / 255.0 
        manual = manual / 255.0
        
        img = np.transpose(img, (2, 0, 1)) # (H, W, C) -> (C, H, W)
        mask = np.expand_dims(mask, axis=0) # (H, W) -> (1, H, W)
        manual = np.expand_dims(manual, axis=0)
        
        # Đóng gói thành Tensor
        img_t = torch.tensor(img, dtype=torch.float32)
        mask_t = torch.tensor(mask, dtype=torch.float32)
        manual_t = torch.tensor(manual, dtype=torch.float32)

        # Trả về 4 outputs nếu có 2nd_manual, ngược lại trả về 3
        if self.manual_paths_2 is not None:
            manual_2 = manual_2 / 255.0
            manual_2 = np.expand_dims(manual_2, axis=0)
            manual_2_t = torch.tensor(manual_2, dtype=torch.float32)
            
            return img_t, mask_t, manual_t, manual_2_t

        return img_t, mask_t, manual_t

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=45, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

val_transform = A.Compose([
    A.Resize(height=256, width=256, p=1.0),
    A.Normalize(p=1.0),
    A.Lambda(mask=normalize_mask, p=1.0),
    ToTensorV2(),
])

In [ ]:
train_dataset = DRIVEDataset(train_image_paths, train_mask_paths, train_manual_paths,transform=train_transform, multiplier=5)
val_dataset = DRIVEDataset(val_image_paths, val_mask_paths, val_manual_paths, transform=val_transform, multiplier=1)
test_dataset = DRIVEDataset(test_image_paths, test_mask_paths, test_manual_1_paths, test_manual_2_paths, transform=val_transform, multiplier=1)

print(f"Train size: {len(train_dataset)}")
print(f"Valid size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
# ===== TEST CLASS DATASET =====
sample_img, sample_mask, sample_manual = train_dataset[0]

print("Dataset check")
print(f"Image tensor shape: {sample_img.shape}")
print(f"Image max/min: {sample_img.max().item()} / {sample_img.min().item()}\n")

print(f"Mask tensor shape: {sample_mask.shape}")
print(f"Mask unique values: {torch.unique(sample_mask).tolist()}\n")

print(f"Manual tensor shape: {sample_manual.shape}")
print(f"Manual unique values: {torch.unique(sample_manual).tolist()}")

Dataset check
Image tensor shape: torch.Size([3, 256, 256])
Image max/min: 1.0 / 0.0
Mask tensor shape: torch.Size([1, 256, 256])
Mask unique values: [0.0, 1.0]


In [ ]:
# ===== AUGMENTATION =====
ori_img_path = train_image_paths[0]
ori_mask_path = train_mask_paths[0]
ori_manual_path = train_manual_paths[0]

ori_img = Image.open(ori_img_path).convert("RGB")
ori_mask = Image.open(ori_mask_path).convert("L")
ori_manual = Image.open(ori_manual_path).convert("L")

aug_img_tensor, aug_mask_tensor, aug_manual_tensor = train_dataset[0]

aug_img = aug_img_tensor.numpy().transpose(1, 2, 0)
aug_mask = aug_mask_tensor.numpy().squeeze()
aug_manual = aug_manual_tensor.numpy().squeeze()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Compare Origin Image and Augmented Image", fontsize=16)

axes[0, 0].imshow(ori_img)
axes[0, 0].set_title("Origin Train Image")
axes[0, 0].axis('off')

axes[0, 1].imshow(ori_mask, cmap='gray')
axes[0, 1].set_title("Origin Mask")
axes[0, 1].axis('off')

axes[0, 2].imshow(ori_manual, cmap='gray')
axes[0, 2].set_title("Origin Manual")
axes[0, 2].axis('off')

axes[1, 0].imshow(aug_img)
axes[1, 0].set_title("Augmented Image")
axes[1, 0].axis('off')

axes[1, 1].imshow(aug_mask, cmap='gray')
axes[1, 1].set_title("Augmented Mask")
axes[1, 1].axis('off')

axes[1, 2].imshow(aug_manual, cmap='gray')
axes[1, 2].set_title("Augmented Manual")
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ===== DATALOADER =====
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Train Loader batch size: {len(train_loader)}")
print(f"Test Loader batch size: {len(test_loader)}")
print(f"Val Loader batch size: {len(val_loader)}")

print("\nDataloader Shape check:")
for images, masks, manual in train_loader:
    print(f"Train Batch - Image shape: {images.shape}")
    print(f"Train Batch - Mask shape: {masks.shape}")
    print(f"Train Batch - Manual shape: {manual.shape}")
    break

In [ ]:
EXPORT_BASE = "/kaggle/working/DRIVE_Export"
os.makedirs(EXPORT_BASE, exist_ok=True)

export_dataset_to_disk(train_dataset, "train")
export_dataset_to_disk(val_dataset, "val")
export_dataset_to_disk(test_dataset, "test")

# ==========================================
# 4. NÉN THƯ MỤC THÀNH FILE ZIP ĐỂ TẢI VỀ
# ==========================================
print("\nCompressing dataset into ZIP file")

ZIP_NAME = "DRIVE_processed_dataset"

ZIP_PATH_BASE = f"/kaggle/working/{ZIP_NAME}" 
shutil.make_archive(ZIP_PATH_BASE, 'zip', EXPORT_BASE)

print(f"Done! Created at: {ZIP_PATH_BASE}.zip")